# 06 Backtesting

This notebook runs the full walk-forward modeling and strategy simulation pipeline, then saves summary tables and charts for strategy evaluation.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.backtester import WalkForwardBacktester
from src.options_strategy import OptionsStrategySelector
from src.signal_model import create_targets
from src.utils import load_yaml_config

config = load_yaml_config(PROJECT_ROOT / "config" / "parameters.yaml")
processed_dir = PROJECT_ROOT / config["data"]["processed_data_dir"]
figures_dir = PROJECT_ROOT / config["reporting"]["figures_dir"]
tables_dir = PROJECT_ROOT / config["reporting"]["tables_dir"]
figures_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)

dataset = pd.read_csv(processed_dir / "nsei_regimes.csv", index_col=0, parse_dates=True)
dataset = create_targets(
    dataset,
    horizon=config["signal_model"]["target_horizon"],
    return_threshold=config["signal_model"]["return_threshold"],
).dropna(subset=["target_binary"])

feature_columns = [column for column in config["signal_model"]["feature_columns"] if column in dataset.columns]
selector = OptionsStrategySelector(**config["options_strategy"])
backtester = WalkForwardBacktester(
    model_name=config["signal_model"]["model_name"],
    task_type=config["signal_model"]["task_type"],
    train_window=config["backtest"]["train_window"],
    test_window=config["backtest"]["test_window"],
    step_size=config["backtest"]["step_size"],
    probability_threshold=config["backtest"]["probability_threshold"],
    short_probability_threshold=config["backtest"]["short_probability_threshold"],
    allow_short=config["backtest"]["allow_short"],
    transaction_cost_bps=config["backtest"]["transaction_cost_bps"],
    slippage_bps=config["backtest"]["slippage_bps"],
    strategy_selector=selector,
)


In [ ]:
result = backtester.run(
    dataset,
    feature_columns=feature_columns,
    target_column="target_binary",
    price_column="close",
    returns_column="daily_return",
)

result.performance_summary.to_csv(tables_dir / "backtest_performance_summary.csv")
result.feature_importance.to_csv(tables_dir / "walk_forward_feature_importance.csv", index=False)

backtester.performance_analyzer.plot_equity_curve(
    result.predictions["strategy_return"],
    benchmark_returns=result.predictions["buy_hold_return"],
    save_path=figures_dir / "equity_curve.png",
)
backtester.performance_analyzer.plot_drawdown(
    result.predictions["strategy_return"],
    save_path=figures_dir / "drawdown_chart.png",
)
backtester.performance_analyzer.plot_rolling_sharpe(
    result.predictions["strategy_return"],
    save_path=figures_dir / "rolling_sharpe.png",
)
backtester.plot_feature_importance(result.feature_importance, save_path=figures_dir / "walk_forward_feature_importance.png")

if "regime_id" in result.predictions.columns:
    backtester.plot_regime_overlay(result.predictions, save_path=figures_dir / "walk_forward_regime_overlay.png")

result.performance_summary
